In [48]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
def get_best_params(model_dict, X_train, y_train, scoring='accuracy', n_iter=100, cv=5):

    random_cv = RandomizedSearchCV(
        estimator=model_dict['model'](),
        param_distributions=model_dict['params'],
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        verbose=3,
        n_iter=n_iter
    )

    num_columns = X_train.select_dtypes(exclude='object').columns
    cat_columns = X_train.select_dtypes('object').columns

    preprocessor = ColumnTransformer(transformers=[
        ('onehot', OneHotEncoder(), cat_columns),
        ('scaler', StandardScaler(), num_columns)
    ])

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('random_model', random_cv)
    ])

    model.fit(X_train, y_train)

    return model.named_steps['random_model'].best_params_

In [49]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

def get_model_score(model_dict, X_train, X_test, y_train, y_test):

    num_columns = X_train.select_dtypes(exclude='object').columns
    cat_columns = X_train.select_dtypes('object').columns

    preprocessor = ColumnTransformer(transformers=[
        ('onehot', OneHotEncoder(), cat_columns),
        ('scaler', StandardScaler(), num_columns)
    ])

    estimator = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model_dict['model'](**model_dict['params']))
    ])

    model_name = model_dict['name']
    estimator.fit(X_train, y_train)

    y_pred_train = estimator.predict(X_train)
    y_pred_test = estimator.predict(X_test)

    # y_train_proba = estimator.predict_proba(X_train)[:, 1]
    # y_test_proba = estimator.predict_proba(X_test)[:, 1]

    # Train performance
    model_accuracy_train = accuracy_score(y_train, y_pred_train)
    model_f1_score_train = f1_score(y_train, y_pred_train, average=None)
    model_precision_train = precision_score(y_train, y_pred_train, average=None)
    model_recall_train = recall_score(y_train, y_pred_train, average=None)
    # model_roc_auc_score_train = roc_auc_score(y_train, y_train_proba)

    # Test performance
    model_accuracy_test = accuracy_score(y_test, y_pred_test)
    model_f1_score_test = f1_score(y_test, y_pred_test, average=None)
    model_precision_test = precision_score(y_test, y_pred_test, average=None)
    model_recall_test = recall_score(y_test, y_pred_test, average=None)
    # model_roc_auc_score_test = roc_auc_score(y_test, y_test_proba)

    print(f" Train Performance for {model_name}")
    print(f"- Accuracy Score: {np.round(model_accuracy_train, 4)}")
    print(f"- F1 Score: {np.round(model_f1_score_train, 4)}")
    print(f"- Precision Score: {np.round(model_precision_train, 4)}")
    print(f"- Recall Score: {np.round(model_recall_train, 4)}")
    # print(f"- ROC AUC Score: {np.round(model_roc_auc_score_train, 4)}")

    print("\n=====================================\n")

    print(f" Test Performance for {model_name}")
    print(f"- Accuracy Score: {np.round(model_accuracy_test, 4)}")
    print(f"- F1 Score: {np.round(model_f1_score_test, 4)}")
    print(f"- Precision Score: {np.round(model_precision_test, 4)}")
    print(f"- Recall Score: {np.round(model_recall_test, 4)}")
    # print(f"- ROC AUC Score: {np.round(model_roc_auc_score_test, 4)}")

    return estimator

In [50]:
import pandas as pd
df = pd.read_csv('du_stats_rca.csv')
print(df.info())
print(df['session_rca'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296208 entries, 0 to 296207
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   ueid         296208 non-null  int64  
 1   session_id   296208 non-null  int64  
 2   avgcqi       296208 non-null  float64
 3   avgmcs       296208 non-null  float64
 4   avgri        296208 non-null  float64
 5   avg_rv0_tx   296208 non-null  float64
 6   tbler        296208 non-null  float64
 7   session_rca  296208 non-null  object 
dtypes: float64(5), int64(2), object(1)
memory usage: 18.1+ MB
None
session_rca
Good UE                191664
MCS Limitation          34848
Channel Quality Bad     34848
No_TX                   34848
Name: count, dtype: int64


In [51]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
X = df.drop(['session_id', 'ueid', 'session_rca', 'tbler'], axis=1)
y = df['session_rca'].map({
    'Good UE': 0,
    'MCS Limitation': 1,
    'No_TX': 2,
    'Channel Quality Bad': 3
})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

model_dict_param_selection = {
    'name': 'Random Forest',
    'model': RandomForestClassifier,
    'params': {
        'n_estimators': [10, 20, 30, 40, 50],
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [1, 2, 3, 4, 5, 10],
        'max_features': [None]
    }
}

model_dict = {
    'name': 'Random Forest',
    'model': RandomForestClassifier,
    'params': get_best_params(
        model_dict=model_dict_param_selection,
        X_train=X_train,
        y_train=y_train,
        scoring='accuracy',
        n_iter=10,
        cv=3
    )
}

estimator = get_model_score(model_dict, X_train, X_test, y_train, y_test)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV 1/3] END criterion=log_loss, max_depth=4, max_features=None, n_estimators=20;, score=0.963 total time=   6.5s
[CV 3/3] END criterion=log_loss, max_depth=4, max_features=None, n_estimators=20;, score=0.963 total time=   6.5s
[CV 2/3] END criterion=log_loss, max_depth=4, max_features=None, n_estimators=20;, score=0.963 total time=   6.5s
[CV 1/3] END criterion=log_loss, max_depth=3, max_features=None, n_estimators=30;, score=0.959 total time=   8.6s
[CV 1/3] END criterion=log_loss, max_depth=1, max_features=None, n_estimators=30;, score=0.764 total time=   3.5s
[CV 2/3] END criterion=log_loss, max_depth=1, max_features=None, n_estimators=30;, score=0.764 total time=   3.9s
[CV 3/3] END criterion=log_loss, max_depth=1, max_features=None, n_estimators=30;, score=0.764 total time=   3.8s
[CV 2/3] END criterion=log_loss, max_depth=3, max_features=None, n_estimators=30;, score=0.960 total time=   8.7s
[CV 3/3] END criterion=log_

In [52]:
features = estimator.named_steps['preprocessor'].get_feature_names_out()
importance = estimator.named_steps['model'].feature_importances_

pd.DataFrame({
    'features': features,
    'importance': importance
})

,features,importance
0,scaler__avgcqi,0.427552
1,scaler__avgmcs,0.251016
2,scaler__avgri,0.005864
3,scaler__avg_rv0_tx,0.315568


In [53]:
ri = [np.round(0 + 4*np.power(np.random.rand(), 1), 2) for _ in range(10)]
cqi = [np.round(0 + 15*np.power(np.random.rand(), 1), 2) for _ in range (10)]
mcs = [np.round(0 + 27*np.power(np.random.rand(), 2), 2) for _ in range(10)]
rv0_tx = [np.round(0 + 20*np.power(np.random.rand(), 1), 2) for _ in range(10)]
data = [(a, b, c, d) for a in ri for b in cqi for c in mcs for d in rv0_tx]
data_new = pd.DataFrame(
    data=data,
    columns=['avgri', 'avgcqi', 'avgmcs', 'avg_rv0_tx']
)
print(data_new.head())
data_new['performance'] = pd.Series(estimator.predict(data_new)).map({
    0: 'Good UE',
    1: 'MCS Limitation',
    2: 'NO TX',
    3: 'Channel Quality Bad'
})
data_new.to_csv('test_data.csv', index=False)

   avgri  avgcqi  avgmcs  avg_rv0_tx
0   0.42    5.58   10.68        7.17
1   0.42    5.58   10.68       14.06
2   0.42    5.58   10.68       18.89
3   0.42    5.58   10.68       15.06
4   0.42    5.58   10.68       14.97
